# Análise de Hábitos de Saúde

**Projeto introdutório de análise de dados com Python**

Este notebook apresenta uma análise exploratória de um conjunto fictício de dados sobre hábitos de saúde de 40 participantes.

## Objetivos da análise
- Avaliar se os participantes dormem dentro da faixa recomendada
- Comparar a frequência de exercícios por sexo
- Investigar a relação entre consumo de água e nível de estresse
- Calcular e classificar o IMC dos participantes

## Ferramentas utilizadas
- `pandas` para leitura e manipulação dos dados
- `matplotlib` para visualização gráfica
- `seaborn` para apoio na estética dos gráficos


## Etapa 1 — Importação das bibliotecas

Nesta etapa, carregamos as bibliotecas que serão utilizadas ao longo da análise.


In [ ]:
# Importação das bibliotecas principais
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações visuais dos gráficos
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='pastel')

print('Bibliotecas importadas com sucesso.')


## Etapa 2 — Carregamento dos dados

Os dados serão lidos a partir de um arquivo CSV e armazenados em um DataFrame chamado `df`.


In [ ]:
# pd.read_csv() lê o arquivo e transforma em tabela
df = pd.read_csv('dados/saude_dados.csv')

# .head(5) mostra as 5 primeiras linhas — útil pra ver se carregou certo
df.head(5)

## Etapa 3 — Conhecimento inicial da base

Antes de gerar indicadores e gráficos, é importante entender a estrutura do conjunto de dados, os tipos das colunas e a distribuição geral das variáveis numéricas.


In [ ]:
# Dimensão da base
linhas, colunas = df.shape
print(f'O dataset possui {linhas} registros e {colunas} colunas.')
print()

# Informações estruturais da base
print('Informações gerais do dataset:')
df.info()


In [ ]:
# .describe() calcula estatísticas básicas: média, mínimo, máximo, etc.
# Só funciona nas colunas numéricas
df.describe().round(2)

## Etapa 4 — Cálculo do IMC

O IMC (Índice de Massa Corporal) será calculado com base no peso e na altura dos participantes.

**Fórmula:** IMC = peso (kg) ÷ altura (m)²


In [ ]:
# Cálculo do IMC
df['imc'] = df['peso_kg'] / ((df['altura_cm'] / 100) ** 2)
df['imc'] = df['imc'].round(1)

# Função para classificação do IMC
def classificar_imc(imc):
    if imc < 18.5:
        return 'Abaixo do peso'
    elif imc < 25:
        return 'Peso normal'
    elif imc < 30:
        return 'Sobrepeso'
    else:
        return 'Obesidade'

df['classificacao_imc'] = df['imc'].apply(classificar_imc)

df[['peso_kg', 'altura_cm', 'imc', 'classificacao_imc']].head()


## Etapa 5 — Visualizações

Nesta etapa, os gráficos ajudam a transformar os números em interpretações mais rápidas e visuais.


In [ ]:
# --- GRÁFICO 1: Distribuição do IMC ---

# Contamos quantas pessoas tem em cada classificação
contagem_imc = df['classificacao_imc'].value_counts()

# Criamos o gráfico de barras
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 2 gráficos lado a lado

# Gráfico de barras (esquerda)
contagem_imc.plot(kind='bar', ax=axes[0], color=['#4CAF50','#FFC107','#FF9800','#F44336'],
                  edgecolor='white', linewidth=1.5)
axes[0].set_title('Classificação do IMC dos Participantes', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classificação')
axes[0].set_ylabel('Quantidade de Pessoas')
axes[0].tick_params(axis='x', rotation=15)

# Gráfico de pizza (direita)
axes[1].pie(contagem_imc.values, labels=contagem_imc.index,
            autopct='%1.1f%%', colors=['#4CAF50','#FFC107','#FF9800','#F44336'],
            startangle=90)
axes[1].set_title('Proporção do IMC', fontsize=14, fontweight='bold')

plt.tight_layout()  # Ajusta o espaçamento entre gráficos
plt.savefig('grafico_imc.png', dpi=150, bbox_inches='tight')  # Salva a imagem
plt.show()
print(f'\n📌 IMC médio: {df["imc"].mean():.1f}')

In [ ]:
# --- GRÁFICO 2: Horas de Sono ---

# A OMS recomenda entre 7 e 9 horas de sono para adultos
sono_ok = df[df['horas_sono'].between(7, 9)].shape[0]   # .between() checa o intervalo
sono_total = df.shape[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma: mostra a distribuição das horas de sono
axes[0].hist(df['horas_sono'], bins=10, color='#5C85D6', edgecolor='white', linewidth=1.5)
axes[0].axvline(7, color='green', linestyle='--', label='Mínimo OMS (7h)')
axes[0].axvline(9, color='red', linestyle='--', label='Máximo OMS (9h)')
axes[0].set_title('Distribuição das Horas de Sono', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Horas de Sono')
axes[0].set_ylabel('Quantidade de Pessoas')
axes[0].legend()

# Comparação: Dentro vs Fora da recomendação
categorias = ['Dentro da\nrecomendação (7-9h)', 'Fora da\nrecomendação']
valores = [sono_ok, sono_total - sono_ok]
axes[1].bar(categorias, valores, color=['#4CAF50', '#F44336'], edgecolor='white', linewidth=1.5)
axes[1].set_title('Sono: Dentro da Recomendação da OMS?', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Quantidade de Pessoas')

plt.tight_layout()
plt.savefig('grafico_sono.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n😴 Média de sono: {df["horas_sono"].mean():.1f}h')
print(f'✅ {sono_ok} de {sono_total} pessoas dormem dentro da recomendação da OMS.')

In [ ]:
# --- GRÁFICO 3: Exercício por Sexo ---

# .groupby() agrupa os dados por categoria
# .mean() calcula a média de cada grupo
exercicio_por_sexo = df.groupby('sexo')['dias_exercicio_semana'].mean().round(1)

fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(['Feminino (F)', 'Masculino (M)'],
                exercicio_por_sexo.values,
                color=['#E91E8C', '#2196F3'], edgecolor='white', linewidth=1.5,
                width=0.5)

# Adiciona o número em cima de cada barra
for barra in barras:
    altura = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., altura + 0.05,
            f'{altura} dias/sem', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Média de Dias de Exercício por Semana (por Sexo)', fontsize=14, fontweight='bold')
ax.set_ylabel('Dias por Semana')
ax.set_ylim(0, 6)

plt.tight_layout()
plt.savefig('grafico_exercicio.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- GRÁFICO 4: Água vs Estresse ---
# Existe relação entre beber mais água e ter menos estresse?

fig, ax = plt.subplots(figsize=(9, 5))

# Scatter plot: cada ponto é uma pessoa
scatter = ax.scatter(df['litros_agua_dia'], df['nivel_estresse'],
                     c=df['nivel_estresse'], cmap='RdYlGn_r',  # verde=baixo, vermelho=alto
                     s=100, alpha=0.7, edgecolors='white')

plt.colorbar(scatter, ax=ax, label='Nível de Estresse')
ax.set_title('Consumo de Água vs Nível de Estresse', fontsize=14, fontweight='bold')
ax.set_xlabel('Litros de Água por Dia')
ax.set_ylabel('Nível de Estresse (1-10)')

plt.tight_layout()
plt.savefig('grafico_agua_estresse.png', dpi=150, bbox_inches='tight')
plt.show()

# .corr() calcula a correlação entre duas colunas
# Varia de -1 a 1: perto de -1 = relação inversa, perto de 0 = sem relação, perto de 1 = relação direta
correlacao = df['litros_agua_dia'].corr(df['nivel_estresse']).round(3)
print(f'🔗 Correlação entre água e estresse: {correlacao}')
print('(Valores próximos de 0 indicam pouca relação linear entre as variáveis)')

## Etapa 6 — Síntese dos resultados

Ao final, consolidamos os principais indicadores gerados ao longo da análise.


In [ ]:
print('=' * 55)
print('        RESUMO DA ANÁLISE DE SAÚDE')
print('=' * 55)
print(f'Total de participantes analisados: {df.shape[0]}')
print()

print('IMC')
print(f'   Média geral: {df["imc"].mean():.1f}')
for classif, qtd in df['classificacao_imc'].value_counts().items():
    print(f'   {classif}: {qtd} pessoas')
print()

print('SONO')
print(f'   Média: {df["horas_sono"].mean():.1f}h por noite')
print(f'   Dentro da recomendação (7 a 9h): {df["horas_sono"].between(7, 9).sum()} pessoas')
print()

print('EXERCÍCIO')
for sexo, media in df.groupby('sexo')['dias_exercicio_semana'].mean().round(1).items():
    nome = 'Feminino' if sexo == 'F' else 'Masculino'
    print(f'   {nome}: {media} dias por semana')
print()

print('ÁGUA E ESTRESSE')
correlacao = df['litros_agua_dia'].corr(df['nivel_estresse'])
print(f'   Correlação entre consumo de água e estresse: {correlacao:.2f}')


---

## Encerramento

Este notebook apresenta uma estrutura simples, mas completa, de análise exploratória com Python.  
Ele pode ser evoluído com novas perguntas de negócio, tratamento de dados faltantes, testes estatísticos e visualizações mais avançadas.

### Próximos passos recomendados
- aplicar filtros e segmentações por perfil dos participantes
- incluir novas variáveis de saúde e comportamento
- testar a análise com bases reais
- publicar o projeto no GitHub como item de portfólio
